In [1]:
%load_ext autoreload
%autoreload 2
import pyfeyngym as pfg

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


# Reduction without cut

In [2]:
IBP_file = "IBP_LI" 
trivial_sector_file = "trivialsector"
modulus = 2**31-1
m_vals = {'d': 73, 'm1': 17, 'm2': 31, 'm3': 53, 'm4': 97}

In [3]:
eq_templates = pfg.gen_eq_templates(IBP_file, m_vals)

In [4]:
# Number of IBP+LI oeprators
len(eq_templates)

18

In [5]:
trivial_sectors = pfg.get_trivial_sectors(trivial_sector_file, cut=[], n_indices=11)

In [6]:
top_sector = (1,1,1,1, 1,1,1,1, 0,0,0)

In [7]:
# Each nontrivial sector is supported on at least one of the followoing spanning cuts.
# Therefore, performing IBP on all spanning cuts allow the complete result,
# i.e. correct coefficients of all masters, to be recovered
spanning_cuts = pfg.spanning_cuts(top_sector, trivial_sectors)
spanning_cuts

[[3, 4, 7],
 [2, 5, 8],
 [2, 5, 7],
 [2, 4, 7],
 [1, 4, 6],
 [2, 4, 6],
 [3, 4, 8],
 [1, 5, 6],
 [3, 5, 8],
 [1, 5, 7],
 [1, 3, 6, 8]]

In [8]:
# Maximum rank, total propagator power, total number of "dots"
s_max, r_max, d_max = 4, 8, 0

In [9]:
all_seeds = pfg.gen_all_seeds(top_sector, trivial_sectors, s_max, r_max, d_max)

In [10]:
len(all_seeds)

32773

In [11]:
improved_seeds = [s for s in all_seeds if
    (pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - 4)) or
                  (s[3]<=0 and s[4]<=0 and s[6]<=0 and pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - 3))]
# The last line makes an exception for sector 167, exactly as in the Kira config file.

In [12]:
len(improved_seeds)

1684

In [13]:
seed_op_eq_list, all_variables = pfg.gen_eqs(eq_templates, trivial_sectors, m_vals, improved_seeds)

In [14]:
# The number of equations is equal to the number of seeds times the number of IBP+LI operators
len(seed_op_eq_list)

30312

In [15]:
equations = [a[-1] for a in seed_op_eq_list]

In [16]:
# Use a built-in sorting function to sort integrals in decreasing complexity
sorted_vars = pfg.sort_integrals_desc(all_variables)
# Below is an alternative, crude one-line implementation
# sorted_vars = sorted(all_variables, key = lambda a: [pfg.t_level(a), pfg.to_sector(a), pfg.d_level(a) + pfg.s_level(a), a], reverse=True)

In [17]:
solution = pfg.solve_eqs_modulo(equations, sorted_vars, modulus)

current row = 24101


In [18]:
# The solution is given as a dictionary
target_integral = (1,1,1,1,1,1,1,1,-3,0,0)
reduced = solution[target_integral]
reduced

[[(1, 1, -1, 0, 1, 1, 0, 1, 0, 0, 0), -253278429],
 [(1, 1, -1, 1, 1, 1, 1, 1, 0, 0, 0), -1198121298],
 [(0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0), 816290119],
 [(1, 1, 1, -1, 1, 0, 1, 1, 0, 0, 0), 1362366730],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0), -2055035966],
 [(1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0), -475074947],
 [(0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0), 621579062],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0), -2126070941],
 [(0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0), 688250012],
 [(1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0), 86939591],
 [(1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0), -417301738],
 [(1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0), 1067794159],
 [(1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0), 401935131],
 [(-1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0), -197040795],
 [(1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0), -1880350946],
 [(0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0), 1995947127],
 [(1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0), -1314232938],
 [(1, 1, 1, 1, 1, -1, 1, 1, 0, 0, 0), 999225869],
 [(1, -1, 1, 0, 1, 0, 1, 1, 0, 0, 0), 1845677127],
 [(0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0), 46178

In [19]:
# The number of terms, i.e. the number of masters, in the above result
len(reduced)

113

In [20]:
# The list here is saved in the "masters" file in this directory
masters = [term[0] for term in reduced]
masters

[(1, 1, -1, 0, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, -1, 1, 1, 1, 1, 1, 0, 0, 0),
 (0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, -1, 1, 0, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0),
 (1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0),
 (0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0),
 (0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0),
 (1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0),
 (1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0),
 (-1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0),
 (0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0),
 (1, 1, 1, 1, 1, -1, 1, 1, 0, 0, 0),
 (1, -1, 1, 0, 1, 0, 1, 1, 0, 0, 0),
 (0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0),
 (1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0),
 (1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1),
 (1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0),
 (1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0),
 (0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0),
 (0, 1, 1, 1, 0

In [21]:
%%time
# Run the solving again, but tracking some information
solution, cost, eqs_order, vars_order = pfg.solve_eqs_modulo(equations, sorted_vars, modulus, return_info=True)

current row = 24101


CPU times: user 9.18 s, sys: 373 ms, total: 9.55 s
Wall time: 9.57 s


In [22]:
# An estimated number of arithmetic operations during solving
cost

42063403

In [23]:
%%time
# With a known list of masters, we can enable the solver to automatically reorder
# the non-master variables to imporove efficiency
# This is done with the `complete_pivoting` flag, which turns on column reordering
# in addition to row reordering. If you want to disable both row and column
# reordering, you can use the `naive_pivoting` flag, used later in the notebook
solution, cost, eqs_order, vars_order = pfg.solve_eqs_modulo(equations, sorted_vars, modulus, return_info=True,
                                     keep_on_rhs = masters, complete_pivoting = True)

current row = 24101


CPU times: user 15.1 s, sys: 1.41 s, total: 16.5 s
Wall time: 16.6 s


In [24]:
cost

40983379

In [25]:
# Total number of equations, number of equations that are non-redundant
len(equations), len(eqs_order)

(30312, 24157)

In [26]:
# A subset of non-redundant equations are given in order used in solving, as indexed my the original equation list
eqs_order[:10]

10-element view(::Vector{Int64}, 1:1:10) with eltype Int64:
 28960
 28996
 29014
 29018
 29036
 29054
 29108
 29126
 29127
 29134

In [27]:
# The variable elimination ordering during solving
vars_order[:10]

10-element view(::Vector{Int64}, 1:1:10) with eltype Int64:
 28212
 28248
 28267
 28373
 28374
 28375
 28378
 28381
 28349
 28380

In [28]:
%%time
# With redundant equation deleted and remaining equations and variables preorderd,
# we can use nopivoting=True to speed up the solving. Here the speedup is not
# significant, but mainly because of the overhead of the Python interface
eqs_preordered = [equations[i-1] for i in eqs_order]
vars_preordered = [sorted_vars[i-1] for i in vars_order]
solution = pfg.solve_eqs_modulo(eqs_preordered, vars_preordered, modulus, nopivoting = True)

current row = 24101


CPU times: user 13 s, sys: 1.27 s, total: 14.3 s
Wall time: 14.3 s


# Reduction on a 4-particle cut

In [29]:
# Now Try IBP on a 4-particle cut [1,3,6,8]. The only code change is
# rebuilding `trivial_sectors` with this cut imposed. Any sector that's
# not supported on the cut is regarded as a trivial sector
trivial_sectors = pfg.get_trivial_sectors(trivial_sector_file, cut=[1,3,6,8], n_indices=11)

In [30]:
s_max, r_max, d_max = 4, 8, 0

In [31]:
all_seeds = pfg.gen_all_seeds(top_sector, trivial_sectors, s_max, r_max, d_max)

In [32]:
len(all_seeds)

2241

In [33]:
improved_seeds = [s for s in all_seeds if
    (pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - 4)) or
                  (s[3]<=0 and s[4]<=0 and s[6]<=0 and pfg.d_level(s) <= 0 and pfg.s_level(s) <= max(1, pfg.t_level(s) - 3))]
# The last line makes an exception for sector 167, exactly as in the Kira config file.

In [34]:
len(improved_seeds)

358

In [35]:
seed_op_eq_list, all_variables = pfg.gen_eqs(eq_templates, trivial_sectors, m_vals, improved_seeds)

In [36]:
len(seed_op_eq_list)

6444

In [37]:
equations = [a[-1] for a in seed_op_eq_list]

In [38]:
sorted_vars = pfg.sort_integrals_desc(all_variables)

In [39]:
solution = pfg.solve_eqs_modulo(equations, sorted_vars, modulus)

current row = 5001


In [40]:
reduced = solution[(1,1,1,1,1,1,1,1,-3,0,0)]
reduced

[[(1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0), 1478300628],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, 0), -929649386],
 [(1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0), 6739694],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0), 92447681],
 [(1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0), 207016776],
 [(1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0), 1672408700],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0), -2126070941],
 [(1, 1, 1, 1, 1, 1, 1, 1, -1, 0, 0), -1585911432],
 [(1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0), 103819801],
 [(1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0), -1079689488],
 [(1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0), -1369706637],
 [(1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0), -1880350946],
 [(1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0), 919757221],
 [(1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0), 1818163544],
 [(1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0), -78710055],
 [(1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0), -1845423702],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1), 1744470477],
 [(1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0), 1157386743],
 [(1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0), 1939518534],
 [(1, 1, 1, 1, 1, 1, 1, 1, 0, -2, 0), -8631

In [41]:
# Number of masters on the 4-particle cut
len(reduced)

27

In [42]:
masters_on_cut = [term[0] for term in reduced]
masters_on_cut

[(1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -1, 0),
 (1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, -1, 0),
 (1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, 0, 0),
 (1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -2, 0, 0),
 (1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 0, 1, 1, 1, 1, -1, 0, 0),
 (1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, -1, 1, 0, 0, 0),
 (1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -1, -1),
 (1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, -1, 1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, -2, 0),
 (1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, 0, 0, -1),
 (1, 1, 1, 1, 0, 1, 1, 1, -1, 0, 0),
 (1, 1, 1, 1, -1, 1, 1, 1, 0, 0, 0),
 (1, 1, 1, 1, 1, 1, 1, 1, -1, 0, -1),
 (1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0),
 (1, -1, 1, 1, 1, 1, 1, 1, 0, 0, 0)]

In [43]:
%%time
solution, cost, _, _ = pfg.solve_eqs_modulo(equations, sorted_vars, modulus, return_info = True)

current row = 5001


CPU times: user 1.17 s, sys: 53.9 ms, total: 1.22 s
Wall time: 1.23 s


In [44]:
cost

4791213

In [45]:
%%time
# Knowing the master list (on the cut), improve efficiency with complete_pivoting
solution, cost, _, _ = pfg.solve_eqs_modulo(equations, sorted_vars, modulus,
                                      keep_on_rhs=masters_on_cut,
                                      complete_pivoting = True,
                                      return_info = True)

current row = 5001


CPU times: user 1.56 s, sys: 55.9 ms, total: 1.62 s
Wall time: 1.62 s


In [46]:
cost

4518657

# Testing computation cost with reordering heuristics disabled
The IBP reduction will strictly adhere to the user-supplied equation and variable ordering.
This may be desirable if you want only AI to do the reordering

In [47]:
%%time
# Test performance with naive pivoting - the cost will be significantly higher!
solution, cost, _, _ = pfg.solve_eqs_modulo(equations, sorted_vars, modulus,
                                      naive_pivoting = True,
                                      return_info = True)

current row = 5001


CPU times: user 11.8 s, sys: 59.9 ms, total: 11.9 s
Wall time: 11.9 s


In [48]:
cost

357518270

In [49]:
%%time
# Finally, `run_ibp_no_reordering` is another useful function.
# It's equivalent to `solve_eqs_modulo` with naive_pivoting=True.
# It is 2.5 times as slow in this case but uses a more memory-efficient
# algorithm. It also checks whether the supplied `target_integral` is
# completely reduced to the supplied masters, after each step of Gaussian
# elimination. The returned cost is slightly lower, as the computation
# is terminated when the reduction is complete, even if some equations
# haven't been used yet.
target_integral = (1,1,1,1,1,1,1,1,0,0,-4)
reduction_complete, cost, n_eqs_used = pfg.run_ibp_no_reordering(equations, sorted_vars,
                                                                 target_integral,
                                                                 masters_on_cut,
                                                                 modulus)

CPU times: user 27.7 s, sys: 185 ms, total: 27.9 s
Wall time: 27.9 s


In [50]:
reduction_complete, cost, n_eqs_used

(True, 357276390, 6076)

In [51]:
# This comparison confirms that the above IBP reduction completed before using all equations
len(equations)

6444